# Debugged VQE Qumode (Stable Version)

Perbaikan utama:
- Menghindari casting complex → float
- Stabilisasi learning rate
- Struktur training loop lebih ringan


In [1]:
import numpy as np
import strawberryfields as sf
import tensorflow as tf
import matplotlib.pyplot as plt

# ── Parameter (sama) ─────────────────────────────────────────────────────────
J      = 1.0
U      = 1.5
cutoff = 8

# ── Benchmark eksak (sama) ────────────────────────────────────────────────────
H_matrix = np.array([
    [U,           J*np.sqrt(2), 0           ],
    [J*np.sqrt(2), 0,           J*np.sqrt(2)],
    [0,           J*np.sqrt(2), U           ]
])
eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)
E0_exact   = eigenvalues[0]
psi0_exact = eigenvectors[:, 0]
print(f"E₀ eksak = {E0_exact:.6f}")
print(f"|ψ₀⟩ eksak = {np.round(psi0_exact, 4)}")

/mgpfs/home/mkhairiansyah/.conda/envs/quantum-md/lib/python3.10/site-packages/strawberryfields/apps/data/sample.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
I0000 00:00:1776347266.440767 1632286 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776347266.445404 1632286 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776347266.502926 1632286 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX

E₀ eksak = -1.386001
|ψ₀⟩ eksak = [-0.4028  0.8219 -0.4028]


In [2]:
# ── Fungsi energi TF (sama, sudah benar) ──────────────────────────────────────
def bose_hubbard_energy_tf(sv_tf, J, U, cutoff):
    energy = tf.constant(0.0, dtype=tf.complex64)
    for i in range(cutoff):
        for j in range(cutoff):
            c_ij = sv_tf[i, j]
            prob = tf.cast(tf.abs(c_ij)**2, tf.complex64)
            onsite = tf.cast((U/2) * (i**2 - i + j**2 - j), tf.complex64)
            energy += prob * onsite
            if i > 0 and j < cutoff - 1:
                energy += tf.cast(J * np.sqrt(i*(j+1)), tf.complex64) \
                          * tf.math.conj(c_ij) * sv_tf[i-1, j+1]
            if i < cutoff - 1 and j > 0:
                energy += tf.cast(J * np.sqrt((i+1)*j), tf.complex64) \
                          * tf.math.conj(c_ij) * sv_tf[i+1, j-1]
    return tf.math.real(energy)

In [3]:
# ── PERBAIKAN ANSATZ ────────────────────────────────────────────────────────
# FIX 1: Inisialisasi di subspace 2-foton (bukan vakum + Dgate)
# FIX 2: Dua layer BSgate untuk expressibility yang cukup
#
# MENGAPA INI BENAR:
# - BSgate adalah gate Gaussian yang KONSERVATIF terhadap jumlah foton
# - Fock(2)|0⟩ + BSgate menjamin semua amplitude di sektor n=2 saja
# - Dua BSgate memberikan 4 parameter untuk 3D subspace → bisa mencapai
#   state apapun dalam subspace 2-foton (up to global phase)

tf.random.set_seed(42)

# 4 parameter untuk 2 layer BSgate
tf_theta1 = tf.Variable(tf.random.normal(shape=[], stddev=0.5))
tf_phi1   = tf.Variable(tf.random.normal(shape=[], stddev=0.5))
tf_theta2 = tf.Variable(tf.random.normal(shape=[], stddev=0.5))
tf_phi2   = tf.Variable(tf.random.normal(shape=[], stddev=0.5))

trainable_params = [tf_theta1, tf_phi1, tf_theta2, tf_phi2]

eng     = sf.Engine(backend="tf", backend_options={"cutoff_dim": cutoff})
circuit = sf.Program(2)

th1, ph1, th2, ph2 = circuit.params("theta1", "phi1", "theta2", "phi2")

with circuit.context as q:
    sf.ops.Fock(2) | q[0]          # ← FIX: start di Fock state |2,0⟩
                                    #   bukan vakum + Dgate
    sf.ops.BSgate(th1, ph1) | (q[0], q[1])  # layer 1: entangle
    sf.ops.BSgate(th2, ph2) | (q[0], q[1])  # layer 2: tambah expressibility

E0000 00:00:1776347270.507948 1632286 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
# ── VQE loop (sama strukturnya) ───────────────────────────────────────────────
opt            = tf.keras.optimizers.Adam(learning_rate=0.05)
steps          = 500   # sedikit lebih banyak untuk konvergensi stabil
energy_history = []
sv = None

print(f"E₀ eksak: {E0_exact:.6f}")

for step in range(steps):
    if eng.run_progs:
        eng.reset()

    with tf.GradientTape() as tape:
        results = eng.run(circuit, args={
            "theta1": tf_theta1, "phi1": tf_phi1,
            "theta2": tf_theta2, "phi2": tf_phi2,
        })
        sv_tf = results.state.ket()
        loss  = bose_hubbard_energy_tf(sv_tf, J, U, cutoff)

    gradients    = tape.gradient(loss, trainable_params)
    grads_n_vars = [(g, v) for g, v in zip(gradients, trainable_params)
                    if g is not None]
    opt.apply_gradients(grads_n_vars)

    E_val = float(loss.numpy())
    sv    = sv_tf.numpy()
    energy_history.append(E_val)

    if step % 60 == 0:
        print(f"Step {step:3d}: E = {E_val:.6f}  (gap = {E_val-E0_exact:.6f})")

print(f"\nHasil akhir:")
print(f"  E_VQE  = {energy_history[-1]:.6f}")
print(f"  E₀_eks = {E0_exact:.6f}")
print(f"  Error  = {abs(energy_history[-1] - E0_exact):.6f}")

E₀ eksak: -1.386001


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# ── Verifikasi konservasi foton (diagnostic baru) ─────────────────────────────
# Cek: apakah semua amplitude memang di sektor n=2?
n_total_prob = sum(
    abs(sv[i,j])**2
    for i in range(cutoff) for j in range(cutoff) if i + j == 2
)
print(f"\nTotal probabilitas di sektor n=2: {n_total_prob:.6f}  (harus = 1.0)")

# Fidelity
fidelity = abs(np.vdot(sv.flatten(),
    np.array([psi0_exact[0] if (i==0 and j==2) else
              psi0_exact[1] if (i==1 and j==1) else
              psi0_exact[2] if (i==2 and j==0) else 0
              for i in range(cutoff) for j in range(cutoff)])
))**2
print(f"Fidelity |⟨ψ_VQE|ψ₀_eks⟩|² = {fidelity:.4f}  (sebelumnya 0.2269)")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(energy_history, lw=2, color='steelblue', label='E(θ) VQE')
ax1.axhline(E0_exact, color='red', ls='--', lw=2, label=f'E₀={E0_exact:.4f}')
ax1.axhline(-2/3,     color='gray', ls=':',  lw=1, label='−0.667 (sebelumnya)')
ax1.set_xlabel('Step')
ax1.set_ylabel('⟨H⟩')
ax1.set_title('Konvergensi VQE (ansatz diperbaiki)')
ax1.legend()

p_vqe   = [abs(sv[0,2])**2, abs(sv[1,1])**2, abs(sv[2,0])**2]
p_exact = [abs(psi0_exact[0])**2, abs(psi0_exact[1])**2, abs(psi0_exact[2])**2]
x = np.arange(3)
ax2.bar(x-0.2, p_exact, 0.38, label='Eksak', color='red',       alpha=0.8)
ax2.bar(x+0.2, p_vqe,   0.38, label='VQE',   color='steelblue', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(['|0,2⟩', '|1,1⟩', '|2,0⟩'])
ax2.set_title('Distribusi probabilitas')
ax2.legend()
plt.tight_layout()
plt.show()